### **[Optimizing ORCL Return-to-Drawdown Performance via Classic and Modified Kaufman’s Adaptive Moving Average (KAMA)](https://algoedgeinsights.beehiiv.com/p/minimizing-false-signals-from-ma-crossovers-with-causal-wiener-deconvolution-walk-forward-optimizati)**

> *Algo-Trading Profitability Analysis via Backtesting & Full-Scale Parameter Optimization without Look Ahead Bias & Overfitting*

In [ ]:
!pip install -qq QuantStats finta ta pandas_ta TA-Lib

In [ ]:
import os
import sys

import warnings
warnings.filterwarnings("ignore")

import math
import requests
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

import plotly.express as px
import plotly.graph_objects as go

import yfinance as yf

import talib
from finta import TA
import quantstats as qs
import pandas_ta as pta
from ta.momentum import kama

USE_EODHD = False
IN_COLAB = 'google.colab' in sys.modules

np.set

if USE_EODHD:
  !pip install -qq eodhd
  from eodhd import APIClient
  SYMBOL = "ORCL.US"
  if IN_COLAB:
    from google.colab import userdata
    API_KEY = userdata.get('EODHD_API_KEY')
  else:
    API_KEY = os.environ.get('EODHD_API_KEY')
else:
  API_KEY = None
  SYMBOL = "ORCL"

In [ ]:
start_date = "2022-01-01"
end_date = "2025-11-29"
interval = '1d'

def fetch_eod(symbol, start, end):
  if USE_EODHD:
    url = f"https://eodhistoricaldata.com/api/eod/{symbol}"
    params = {"from": start, "to": end, "fmt": "json", "api_token": API_KEY}
    r = requests.get(url, params=params)
    r.raise_for_status()
    df = pd.DataFrame(r.json())
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date').sort_index()

    if "adjusted_close" in df.columns:
        df["adj_close"] = df["adjusted_close"]
    else:
        df["adj_close"] = df["close"]
    return df[['open','high','low','close','adj_close','volume']]
  else:
    df = yf.download(symbol, start=start, end=end, multi_level_index=False, progress=False, interval=interval)
    df = df[['Open', 'High', 'Low', 'Close']]
    return df

In [ ]:
symbol = "ORCL"
TRADING_DAYS = 30

if USE_EODHD:
  api = APIClient(API_KEY)
  resp = api.get_eod_historical_stock_market_data(symbol=symbol, period='d', from_date=start_date, to_date=end_date, order='a')
  dfvx = pd.DataFrame(resp)
  df = dfvx.copy()
  df['date'] = pd.to_datetime(df['date'])
  df = df.sort_values('date')
  df.set_index('date', inplace=True)
  df = df[['open', 'high', 'low', 'close']]
else:
  df = yf.download(symbol, start=start_date, end=end_date, multi_level_index=False, progress=False, interval=interval)
  df = df[['Open', 'High', 'Low', 'Close']]

In [ ]:
df.info()

In [ ]:
display(df.head(10))

In [ ]:
display(df.tail(10))

In [ ]:
end = datetime.utcnow().date()
start = end - timedelta(days=YEARS*365)

print("Fetching data...")
df = fetch_eod(SYMBOL, start.isoformat(), end.isoformat(), API_TOKEN)

df.info()

In [ ]:
# ---- Build figure ----
fig = go.Figure()

# Candlestick chart
fig.add_trace(
  go.Candlestick(
    x=df.index,
    open=df["open"],
    high=df["high"],
    low=df["low"],
    close=df["close"],
    name="ORCL Candlesticks"
  )
)

# Volume bars (secondary y-axis)
fig.add_trace(
  go.Bar(
    x=df.index,
    y=df["volume"],
    name="Volume",
    marker_opacity=1.0,  marker=dict(color="darkblue"),   # change color
    yaxis="y2"
  )
)

# Layout with two y-axes
fig.update_layout(
  title=f"{SYMBOL} — Candlestick & Volume",
  xaxis_title="Date",
  yaxis_title="Price",

  yaxis2=dict(
    title="Volume",
    overlaying="y",
    side="right",
    showgrid=True,
  ),

  xaxis_rangeslider_visible=False,
  height=600
)

fig.show()

In [ ]:
# Function to detect bullish engulfing
def bullish_engulfing(df):
  pattern = []
  for i in range(1, len(df)):
    prev = df.iloc[i-1]
    curr = df.iloc[i]
    # Bullish Engulfing: current green candle engulfs previous red candle
    if (prev['close'] < prev['open']) and (curr['close'] > curr['open']) and (curr['open'] < prev['close']) and (curr['close'] > prev['open']):
      pattern.append(i)
  return pattern

# Function to detect hammer
def hammer(df):
  pattern = []
  for i in range(len(df)):
    candle = df.iloc[i]
    body = abs(candle['close'] - candle['open'])
    lower_shadow = candle['open'] - candle['low'] if candle['close'] > candle['open'] else candle['close'] - candle['low']
    upper_shadow = candle['high'] - candle['close'] if candle['close'] > candle['open'] else candle['high'] - candle['open']
    if lower_shadow >= 2 * body and upper_shadow <= body:
      pattern.append(i)
  return pattern

bullish = bullish_engulfing(df)
hammers = hammer(df)

#Plotting

fig = go.Figure()

# Candlestick
fig.add_trace(go.Candlestick(
  x=df.index,
  open=df['open'],
  high=df['high'],
  low=df['low'],
  close=df['close'],
  name='ORCL Price'
))

# Bullish Engulfing markers
fig.add_trace(go.Scatter(
  x=df.index[bullish],
  y=df['high'][bullish] + 1,  # slightly above high
  mode='markers',
  marker=dict(symbol='triangle-up', color='green', size=14),
  name='Bullish Engulfing'
))

# Hammer markers
fig.add_trace(go.Scatter(
  x=df.index[hammers],
  y=df['low'][hammers] - 1,  # slightly below low
  mode='markers',
  marker=dict(symbol='circle', color='orange', size=12),
  name='Hammer'
))

# Layout
fig.update_layout(
  title='ORCL Candlestick Chart with Key Patterns',
  xaxis_title='Date',
  yaxis_title='Price',height=700,
  xaxis_rangeslider_visible=True
)

fig.show()

In [ ]:
df_orcl = fetch_eod('ORCL', start.isoformat(), end.isoformat())
df_msft = fetch_eod('MSFT', start.isoformat(), end.isoformat())
df_amzn = fetch_eod('AMZN', start.isoformat(), end.isoformat())
df_spy = fetch_eod('SPY', start.isoformat(), end.isoformat())

In [ ]:
df_orcl['daily_return'] = df_orcl['close'].pct_change()[1:]
df_msft['daily_return'] = df_msft['close'].pct_change()[1:]

df_spy['daily_return'] = df_spy['close'].pct_change()[1:]

threshold=-0.2
amzn_daily_return = df_amzn['daily_return'][df_amzn['daily_return'] > threshold]

In [ ]:
df_orcl['cum_return'].plot(label='ORCL')
df_msft['cum_return'].plot(label='MSFT')

amzn_cum_return.plot(label='AMZN')
df_spy['cum_return'].plot(label='SPY')

plt.title('Cumulative Returns')
plt.legend()
plt.show()

In [ ]:
returns = np.log(df_orcl['close']/df_orcl['close'].shift(1))
returns.fillna(0, inplace=True)
volatility = returns.rolling(window=TRADING_DAYS).std() * np.sqrt(TRADING_DAYS)
sharpe_ratio_orcl = returns.mean()/volatility
sharpe_ratio_orcl.tail()

In [ ]:
returns = np.log(df_amzn['close']/df_amzn['close'].shift(1))
returns.fillna(0, inplace=True)
volatility = returns.rolling(window=TRADING_DAYS).std() * np.sqrt(TRADING_DAYS)
sharpe_ratio_amzn = returns.mean() / volatility

In [ ]:
returns = np.log(df_spy['close']/df_spy['close'].shift(1))
returns.fillna(0, inplace=True)
volatility = returns.rolling(window=TRADING_DAYS).std() * np.sqrt(TRADING_DAYS)
sharpe_ratio_spy = returns.mean() / volatility

In [ ]:
returns = np.log(df_msft['close']/df_msft['close'].shift(1))
returns.fillna(0, inplace=True)
volatility = returns.rolling(window=TRADING_DAYS).std() * np.sqrt(TRADING_DAYS)
sharpe_ratio_msft = returns.mean() / volatility

In [ ]:
fig = plt.figure(figsize=(15, 7))
ax3 = fig.add_subplot(1, 1, 1)
sharpe_ratio.plot(ax=ax3,label='ORCL')
sharpe_ratio_amzn.plot(ax=ax3,label='AMZN')
sharpe_ratio_msft.plot(ax=ax3,label='MSFT')
sharpe_ratio_spy.plot(ax=ax3,label='SPY')
ax3.set_xlabel('Date')
ax3.set_ylabel('Sharpe Ratio')
ax3.legend()
ax3.set_title('Rolling 30-Day Sharpe Ratio')
plt.show()

In [ ]:
# extend pandas functionality with metrics, etc.
qs.extend_pandas()

# fetch the daily returns for a stock
ticker_list=['ORCL','MSFT','AMZN','SPY']
stock = qs.utils.download_returns(ticker_list,period="5y")

# show sharpe ratio
qs.stats.sharpe(stock)

In [ ]:
# or using extend_pandas()
stock.sharpe()

In [ ]:
stock.sharpe().plot.bar(title='Sharpe Ratio')
plt.show()

In [ ]:
stock.avg_loss().plot.bar(title='Average Loss')
plt.show()

In [ ]:
stock.avg_return().plot.bar(title='Average Return')
plt.show()

In [ ]:
stock.cagr().plot.bar(title='CAGR')
plt.show()

In [ ]:
tickers = ['AMZN', 'MSFT', 'ORCL', 'SPY']

plt.bar(tickers, cvar)
plt.title('CVaR')
plt.xlabel('Ticker')
plt.ylabel('CVaR')
plt.show()

In [ ]:
qs.stats.sharpe(stock)

In [ ]:
qs.stats.avg_loss(stock)

In [ ]:
qs.stats.avg_return(stock)

In [ ]:
qs.stats.kelly_criterion(stock)

In [ ]:
qs.stats.gain_to_pain_ratio(stock)

In [ ]:
qs.stats.max_drawdown(stock)

In [ ]:
qs.stats.risk_return_ratio(stock)

In [ ]:
qs.stats.kurtosis(stock)

In [ ]:
qs.stats.skew(stock)

In [ ]:
qs.stats.sortino(stock)

In [ ]:
qs.stats.tail_ratio(stock)

In [ ]:
qs.stats.ulcer_performance_index(stock)

In [ ]:
qs.stats.win_loss_ratio(stock)

In [ ]:
qs.stats.win_rate(stock)

In [ ]:
impl_vol = qs.stats.implied_volatility(stock)

impl_vol['SPY'].plot(label='SPY')
impl_vol['ORCL'].plot(label='ORCL')
impl_vol['MSFT'].plot(label='MSFT')
impl_vol['AMZN'].plot(label='AMZN')
plt.legend()
plt.title('Implied Volatility')
plt.show()

In [ ]:
# Compute correlation matrix
co_mtx = implvol.corr(numeric_only=True)

# Print correlation matrix
display(co_mtx)

# Plot correlation heatmap
sns.heatmap(co_mtx, cmap="YlGnBu", annot=True)

# Display heatmap
plt.show()

In [ ]:
# KAMA params
ER_PERIOD = 10       # Efficiency Ratio period
FAST_SC_FAST = 5     # used to compute fast smoothing constant (typical: 2 -> fastest)
FAST_SC_SLOW = 10    # used to compute fast smoothing constant (typical: 30 -> slowest)

In [ ]:
# Monkey-patch Pandas to add iteritems back
if not hasattr(pd.Series, "iteritems"):
  pd.Series.iteritems = pd.Series.items

kama_period = ER_PERIOD
df["KAMAFTA"] = TA.KAMA(df, kama_period)
plt.figure(figsize=(11,5))
plt.plot(df.index, df["close"], label="Close")
plt.plot(df.index, df["KAMAFTA"], label=f"KAMA({kama_period})")
plt.title("Close price & KAMA FINTA")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Create KAMA indicator (default window=10, pow1=2, pow2=30)
kamata = kama(close=df["close"], window=ER_PERIOD, pow1=FAST_SC_FAST, pow2=FAST_SC_SLOW)

# Add KAMA to dataframe
df["KAMATA"] = kamata

# Plotting comparison KAMA TA vs FinTA
plt.figure(figsize=(14,6))
df["KAMATA"].plot(label='TA')
df["KAMAFTA"].plot(label='FinTA')
plt.grid()
plt.legend()
plt.title('ORCL KAMA Indicator')
plt.show()

In [ ]:
df["KAMAPTA"] = pta.kama(df["Close"], length=ER_PERIOD, fast=FAST_SC_FAST, slow=FAST_SC_SLOW)
display(df[["KAMATA", "KAMAPTA"]].tail())

In [ ]:
df["KAMAtalib"] = talib.KAMA(df["Close"], timeperiod=ER_PERIOD)
display(df[["KAMAFTA", "KAMAtalib"]].tail())

In [ ]:
def kama_classic(price, er_period=10, fast=2, slow=30):
  price = price.astype(float)
  change = abs(price.diff(er_period))
  volatility = price.diff().abs().rolling(er_period).sum()

  ER = change / volatility
  fastSC = 2 / (fast + 1)
  slowSC = 2 / (slow + 1)

  SC = (ER * (fastSC - slowSC) + slowSC) ** 2

  kama = pd.Series(np.nan, index=price.index)
  kama.iloc[er_period] = price.iloc[er_period]

  for i in range(er_period + 1, len(price)):
    kama.iloc[i] = kama.iloc[i-1] + SC.iloc[i] * (price.iloc[i] - kama.iloc[i-1])
  return kama

price = df['Close']
df['kama_class'] = kama_classic(price, er_period=ER_PERIOD, fast=FAST_SC_FAST, slow=FAST_SC_SLOW)
display(df[["KAMATA", "kama_class"]].tail())

In [ ]:
def kama_log_er(price, er_period=10, fast=2, slow=30):
  logp = np.log(price)

  change = abs(logp.diff(er_period))
  volatility = logp.diff().abs().rolling(er_period).sum()
  ER = change / volatility

  fastSC = 2 / (fast + 1)
  slowSC = 2 / (slow + 1)
  SC = (ER * (fastSC - slowSC) + slowSC) ** 2

  kama = pd.Series(np.nan, index=price.index)
  kama.iloc[er_period] = price.iloc[er_period]

  for i in range(er_period + 1, len(price)):
    kama.iloc[i] = kama.iloc[i-1] + SC.iloc[i] * (price.iloc[i] - kama.iloc[i-1])
  return kama

df['kama_logret'] = kama_log_er(price, er_period=ER_PERIOD, fast=FAST_SC_FAST, slow=FAST_SC_SLOW)
display(df[["kama_class", "kama_logret"]].tail())

In [ ]:
def kama_linear_sc(price, er_period=10, fast=2, slow=30):
  change = abs(price.diff(er_period))
  volatility = price.diff().abs().rolling(er_period).sum()
  ER = change / volatility

  fastSC = 2 / (fast + 1)
  slowSC = 2 / (slow + 1)

  SC = ER * (fastSC - slowSC) + slowSC   # <-- linear, not squared

  kama = pd.Series(np.nan, index=price.index)
  kama.iloc[er_period] = price.iloc[er_period]

  for i in range(er_period + 1, len(price)):
    kama.iloc[i] = kama.iloc[i-1] + SC.iloc[i] * (price.iloc[i] - kama.iloc[i-1])
  return kama

df['kama_linsc'] = kama_linear_sc(price, er_period=ER_PERIOD, fast=FAST_SC_FAST, slow=FAST_SC_SLOW)

# Plotting
plt.figure(figsize=(14,6))
df['Close'].plot(label='Close',alpha=0.5)
df['kama_linsc'].plot(label='Linear SC', lw=2)
df['kama_class'].plot(label='Classic', alpha=0.6)
plt.legend()
plt.ylabel('Price USD')
plt.title('ORCL KAMA Indicator vs Close Price')
plt.grid()
plt.show()

In [ ]:
def kama_sqrt_sc(price, er_period=10, fast=2, slow=30):
  change = abs(price.diff(er_period))
  volatility = price.diff().abs().rolling(er_period).sum()
  ER = change / volatility

  fastSC = 2 / (fast + 1)
  slowSC = 2 / (slow + 1)

  SC = np.sqrt(ER * (fastSC - slowSC) + slowSC)

  kama = pd.Series(np.nan, index=price.index)
  kama.iloc[er_period] = price.iloc[er_period]

  for i in range(er_period + 1, len(price)):
    kama.iloc[i] = kama.iloc[i-1] + SC.iloc[i] * (price.iloc[i] - kama.iloc[i-1])
  return kama

df['kama_sqrtsc'] = kama_sqrt_sc(price, er_period=ER_PERIOD, fast=FAST_SC_FAST, slow=FAST_SC_SLOW)

# Plotting
plt.figure(figsize=(14,6))
df['Close'].plot(label='Close',alpha=0.5)
df['kama_sqrtsc'].plot(label='SQRT SC',lw=2)
df['kama_linsc'].plot(label='Linear SC',lw=1)
plt.legend()
plt.ylabel('Price USD')
plt.title('ORCL Close Price vs SQRT/Linear SC KAMA Indicator')
plt.grid()
plt.show()

In [ ]:
def kama_clipped_sc(price, er_period=10, fast=2, slow=30, minSC=0.01, maxSC=0.5):
  change = abs(price.diff(er_period))
  volatility = price.diff().abs().rolling(er_period).sum()
  ER = change / volatility

  fastSC = 2 / (fast + 1)
  slowSC = 2 / (slow + 1)

  SC = (ER * (fastSC - slowSC) + slowSC) ** 2
  SC = SC.clip(minSC, maxSC)

  kama = pd.Series(np.nan, index=price.index)
  kama.iloc[er_period] = price.iloc[er_period]

  for i in range(er_period + 1, len(price)):
    kama.iloc[i] = kama.iloc[i-1] + SC.iloc[i] * (price.iloc[i] - kama.iloc[i-1])
  return kama

df['kama_clippedsc'] = kama_clipped_sc(price, er_period=ER_PERIOD, fast=FAST_SC_FAST, slow=FAST_SC_SLOW,minSC=0.1, maxSC=0.3)

# Plotting
plt.figure(figsize=(14,6))
df['Close'].plot(label='Close',alpha=0.5)
df['kama_class'].plot(label='Classic',alpha=0.6)
df['kama_clippedsc'].plot(label='Clipped SC',lw=2)
plt.legend()
plt.ylabel('Price USD')
plt.title('ORCL KAMA Classic/Clipped SC Indicator vs Close Price')
plt.grid()
plt.show()

In [ ]:
def kama_zero_lag(price, er_period=10, fast=2, slow=30):
  kama1 = kama_classic(price, er_period, fast, slow)
  kama2 = kama_classic(kama1.dropna(), er_period, fast, slow)
  kama2 = kama2.reindex(price.index)

  return kama1 + (kama1 - kama2)

df['kama_zerolag'] = kama_zero_lag(price, er_period=ER_PERIOD, fast=FAST_SC_FAST, slow=FAST_SC_SLOW)

# Plotting
plt.figure(figsize=(14,6))
df['Close'].plot(label='Close', alpha=0.5)
df['kama_linsc'].plot(label='Linear SC', lw=1)
df['kama_class'].plot(label='Classic', alpha=0.6)
df['kama_zerolag'].plot(label='ZeroLag', lw=2)
plt.legend()
plt.ylabel('Price USD')
plt.title('ORCL KAMA Classic/Linear SC/Zero Lag Indicator vs Close Price')
plt.grid()
plt.show()

In [ ]:
period = 10
period_fast = 2
period_slow = 15
period_fast1 = 5

close = df['Close']
kama2 = kama_classic(close, period, period_fast, period_slow)
kama5 = kama_classic(close, period, period_fast1, period_slow)
df['kama5'] = kama5
df['kama2'] = kama2
kama_roc = df['kama5'].pct_change()

# Calculate the signal
df['signal'] = np.where((kama_roc > 0) & (close >= kama2), 1, np.where((kama_roc < 0) & (close <= kama2), -1, 0))

df['signal_shifted'] = df['signal'].shift()

buysignals = df[df["signal_shifted"] == 1]
sellsignals = df[df["signal_shifted"] == -1]

# Calculate the returns on the days we trigger a signal
df['returns'] = df['Close'].pct_change()

# Calculate the strategy returns
df['strategy_returns'] = df['signal_shifted'] * df['returns']

# Calculate the cumulative returns
df1 = df.dropna()
df1['cumulative_returns'] = (1 + df1['strategy_returns']).cumprod()
df1['bh_cumulative_returns'] = (1 + df['returns']).cumprod()

df_concat = pd.concat((df[['Close','kama2','kama5','signal']], df1[['cumulative_returns']]), axis=1)

fig, (ax0,ax1) = plt.subplots(nrows=2, sharex=True, subplot_kw=dict(frameon=True),figsize=(12,14))

df_concat[['close','kama2','kama5']].plot(ax=ax0)
ax0.set_title("KAMA")
ax0.set_ylabel('Price USD')
ax0.grid()


df_concat[['cumulative_returns']].plot(ax=ax1)
df1['bh_cumulative_returns'].plot(ax=ax1)
ax1.legend()
ax1.set_title("Cumulative Returns: Strategy vs Buy&Hold (BH)")
ax1.set_ylabel('Cumulative Return')
ax1.grid()
plt.show()

In [ ]:
def compute_metrics_from_returns(df,myreturns):
  ret = df[myreturns]

  # Equity curve
  equity = (1 + ret).cumprod()

  # Total Return
  total_return = equity.iloc[-1] - 1

  # Years
  years = (df.index[-1] - df.index[0]).days / 365.25

  # CAGR
  cagr = (equity.iloc[-1]) ** (1 / years) - 1

  # Volatility (annualized)
  vol = ret.std() * np.sqrt(252)

  # Sharpe ratio (rf=0)
  sharpe = cagr / vol

  # Downside volatility for Sortino
  downside = ret[ret < 0]
  sortino = cagr / (downside.std() * np.sqrt(252))

  # Max drawdown
  rolling_peak = equity.cummax()
  drawdown = (equity - rolling_peak) / rolling_peak
  max_dd = drawdown.min()

  # Calmar ratio
  calmar = cagr / abs(max_dd)

  return {
    'Total Return': total_return,
    'CAGR': cagr,
    'Volatility': vol,
    'Sharpe': sharpe,
    'Sortino': sortino,
    'Max Drawdown': max_dd,
    'Calmar': calmar
  }

In [ ]:
# Strategy
metrics = compute_metrics_from_returns(df,'strategy_returns')
for k, v in metrics.items():
  print(f"{k:<15}: {v:.4f}")

In [ ]:
# Buy&Hold
metrics = compute_metrics_from_returns(df,'returns')
for k, v in metrics.items():
  print(f"{k:<15}: {v:.4f}")

In [ ]:
plt.figure(figsize=(14,6))

plt.plot(df.index,df.close)

for idx in buysignals.index.tolist():
  plt.plot(
    idx,
    df.loc[idx]["Close"],
    "g*",
    markersize=8
  )

for idx in sellsignals.index.tolist():
  plt.plot(
    idx,
    df.loc[idx]["Close"],
    "r*",
    markersize=8
  )
plt.title('ORCL CLose Price: Bullish (Green) & Bearish (Red) Trends')
plt.xlabel('Date')
plt.ylabel('Price USD')
plt.grid()
plt.show()

In [ ]:
def kama_strategy_backtest(close, fast_period, slow_period):
  df = pd.DataFrame({'Close': close})
  df['pct_change'] = df['Close'].pct_change()
  df['fast_sma'] = talib.KAMA(df["Close"], timeperiod=fast_period)
  df['slow_sma'] = talib.KAMA(df["Close"], timeperiod=slow_period)

  # Generate signal
  df['signal'] = 0
  df.loc[(df['fast_sma'] > df['slow_sma']), 'signal'] = 1
  df.loc[(df['fast_sma'] < df['slow_sma']), 'signal'] = -1

  # Calculate returns with shift to avoid lookahead bias
  df['strategy_return'] = df['pct_change'] * df['signal'].shift(1)
  df['equity'] = 100 * (1 + df['strategy_return']).cumprod()

  # Calculate Buy and Hold total return in percentage
  df['bnh_equity'] = 100 * (1 + df['pct_change']).cumprod()
  bnh_total_ret = (df['bnh_equity'].iloc[-1] / df['bnh_equity'].dropna().iloc[0] - 1) * 100

  # Strategy total return
  equity = df['equity']
  total_ret = (equity.iloc[-1] / equity.dropna().iloc[0] - 1) * 100

  return equity, total_ret, bnh_total_ret

fast_period = 5
slow_period = 10

price = df.Close

equity,total_ret, bnh_total_ret = kama_strategy_backtest(price, fast_period, slow_period)

print("Total return (%):", total_ret)
print("Buy and Hold return (%):", bnh_total_ret)

In [ ]:
def optimize_kama_periods(close, fast_range, slow_range):
  best_result = {'fast': None, 'medium': None, 'slow': None, 'total_return': -np.inf}
  best_equity = None

  # Iterate valid combinations: fast < medium < slow
  for fast, slow in product(fast_range, slow_range):
    if fast < slow:
      equity, total_ret, bnh_total_ret = kama_strategy_backtest(close, fast, slow)
      if total_ret > best_result['total_return']:
        best_result = {'fast': fast, 'slow': slow, 'total_return': total_ret}
        best_equity = equity
        buy_and_hold = bnh_total_ret

  return {
    'best_periods': (best_result['fast'], best_result['slow']),
    'best_total_return': best_result['total_return'],
    'best_equity': best_equity,
    'buy_and_hold': buy_and_hold
  }

fast_range = range(5, 46, 5)
slow_range = range(10, 150, 10)

result = optimize_kama_periods(price, fast_range, slow_range)
print("Best periods (fast, slow):", result['best_periods'])
print("Best total return (%):", result['best_total_return'])
print("Buy and Hold return (%):", result['buy_and_hold'])

In [ ]:
# Train Dataset
from_date_train = '2022-01-03'
to_date_train = '2025-04-30'

df_train = df.loc[from_date_train:to_date_train]

result = optimize_kama_periods(df_train['Close'], fast_range, slow_range)

best_fast = result['best_periods'][0]
best_slow = result['best_periods'][1]

print("Best periods (fast, slow):", best_fast, best_slow)
print("Best total return (%):", result['best_total_return'])
print("Buy and Hold return (%):", result['buy_and_hold'])

In [ ]:
# Test Dataset
from_date_test = '2025-05-01'
to_date_test = '2025-11-26'

df_test = df.loc[from_date_test:to_date_test]
equity,total_ret, bnh_total_ret = kama_strategy_backtest(df_test['Close'], best_fast, best_slow)

print("Best periods applied (fast, medium, slow):", best_fast, best_slow)
print("Total return (%):", total_ret)
print("Buy and Hold return (%):", bnh_total_ret)

In [ ]:
close_prices = df['Close']

def non_parametric_brownian_bridge(close_prices, n_paths=1000, seed=42):
  np.random.seed(seed)
  n = len(close_prices)
  X0 = np.log(close_prices.iloc[0])
  Xn = np.log(close_prices.iloc[-1])
  log_returns = np.log(close_prices / close_prices.shift(1)).dropna().values

  paths = np.zeros((n, n_paths))
  for i in range(n_paths):
    # Sample n-1 returns and center them
    sampled = np.random.choice(log_returns, size=n - 1, replace=True)
    drift_correction = (Xn - X0) / (n - 1) - np.mean(sampled)
    sampled += drift_correction  # Center drift
    W = np.concatenate(([0], np.cumsum(sampled)))  # Now length n

    # Brownian bridge formula for all time steps (n)
    bridge = X0 + W + np.linspace(0, 1, n) * (Xn - X0 - W[-1])
    paths[:, i] = bridge

  sim_prices = np.exp(paths)
  sim_prices[~np.isfinite(sim_prices)] = np.nan
  return sim_prices


simulated_paths = non_parametric_brownian_bridge(close_prices, n_paths=1000)

for i in range(simulated_paths.shape[1]):
  df[f'sim_path_{i+1}'] = simulated_paths[:, i]

plt.figure(figsize=(14, 7))
plt.plot(df.index, df.loc[:, 'sim_path_1':'sim_path_1000'], lw=1, alpha=0.7)
plt.plot(df.index, close_prices, lw=2, label='Original', color='black')
plt.title('Non-Parametric Brownian Bridge - Simulated Paths')
plt.xlabel('Date')
plt.ylabel('Price')
plt.grid()
plt.legend()
plt.show()

In [ ]:
df_train_multiple_paths = df.loc[from_date_train:to_date_train]

results = []

for i in tqdm(range(1,1001,1)):
    print(f'Processing path {i}')
    for fast, slow in product(fast_range, slow_range):
        _, total_backtest_ret, _  = sma_strategy_backtest(df_train_multiple_paths['sim_path_' + str(i)], fast, slow)
        result = {'fast': fast, 'slow': slow, 'total_return': total_backtest_ret}
        results.append(result)

df_all_paths_train = pd.DataFrame(results)
if not os.path.exists('df_all_paths_train.csv') and not IN_COLAB:
  df_all_paths_train.to_csv('df_all_paths_train.csv', index=False)

display(df_all_paths_train.sample(25))

In [ ]:
unique_combos = (
  df_all_paths_train[['fast', 'slow']]
  .drop_duplicates()
  .copy()
)

unique_combos[['fast', 'slow']] = unique_combos[['fast', 'slow']].astype(int)

def _compute_test_metrics(row):
  f, s = int(row['fast']), int(row['slow'])
  _, total_ret, bnh_ret = sma_strategy_backtest(df_test['close'], f, s)
  return pd.Series({'test_total_return': total_ret, 'test_bnh_return': bnh_ret})

# Evaluate each unique combo once
unique_combos[['test_total_return', 'test_bnh_return']] = unique_combos.apply(_compute_test_metrics, axis=1)

# Join back to all rows to align with every path's chosen combo
df_all_paths_with_test = df_all_paths_train.merge(unique_combos, on=['fast', 'slow'], how='left')
display(df_all_paths_with_test.sample(25))

In [ ]:
plt.scatter(df_all_paths_with_test['fast'],
                     df_all_paths_with_test['slow'],c=df_all_paths_with_test['test_total_return'],
                     cmap='viridis', s=df_all_paths_with_test['test_total_return']*5,
                     alpha=0.7)
plt.xlabel('Fast KAMA Length')
plt.ylabel('Slow KAMA Length')
plt.title('Test Total Return')
plt.colorbar()
plt.grid()
plt.show()

In [ ]:
plt.scatter(df_all_paths_with_test['fast'],
                     df_all_paths_with_test['slow'],c=-df_all_paths_with_test['test_total_return'],
                     cmap='viridis', s=-df_all_paths_with_test['test_total_return']*5,
                     alpha=0.7)
plt.xlabel('Fast KAMA Length')
plt.ylabel('Slow KAMA Length')
plt.title('-(Test Total Return)')
plt.colorbar()
plt.grid()
plt.show()

In [ ]:
fast_bins = np.arange(df_all_paths_with_test['fast'].min(),
                      df_all_paths_with_test['fast'].max() + 5, 5)
slow_bins = np.arange(df_all_paths_with_test['slow'].min(),
                      df_all_paths_with_test['slow'].max() + 10, 10)

# Create binned columns for fast and slow
df_all_paths_with_test['fast_bin'] = pd.cut(df_all_paths_with_test['fast'], fast_bins)
df_all_paths_with_test['slow_bin'] = pd.cut(df_all_paths_with_test['slow'], slow_bins)

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

# Boxplot for fast parameter bins
df_all_paths_with_test.boxplot(column='test_total_return', by='fast_bin', ax=axes[0], grid=False)
axes[0].set_title('Test Returns by Fast KAMA Length')
axes[0].set_xlabel('Fast KAMA Length Range')
axes[0].set_ylabel('Test Period Return')
axes[0].tick_params(axis='x', rotation=45)

# Boxplot for slow parameter bins
df_all_paths_with_test.boxplot(column='test_total_return', by='slow_bin', ax=axes[1], grid=False)
axes[1].set_title('Test Returns by Slow KAMA Length')
axes[1].set_xlabel('Slow KAMA Length Range')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('')  # Remove default pandas title
plt.tight_layout()
plt.show()

In [ ]:
df_all_paths_with_test['overfit'] = df_all_paths_with_test['total_return'] - df_all_paths_with_test['test_total_return']

# Group by fast and slow and aggregate overfit by mean (or median if preferred)
agg_df = df_all_paths_with_test.groupby(['fast', 'slow'])['overfit'].mean().reset_index()

# Pivot the aggregated DataFrame
heatmap_data = agg_df.pivot(index='fast', columns='slow', values='overfit')

plt.figure(figsize=(12, 8))
sns.heatmap(abs(heatmap_data), cmap='coolwarm', center=0,
            cbar_kws={'label': 'Overfitting Risk (Train - Test Return)'},
            linewidths=0.5)

plt.title('Heatmap of Overfitting Risk by KAMA Parameters')
plt.xlabel('Slow KAMA Length')
plt.ylabel('Fast KAMA Length')
plt.show()